# Stock Market Direction Prediction — Logistic Regression

**Student ID:** IT25102978  
**Model:** Logistic Regression  
**Task:** Binary classification — predict whether the next closing price moves up (`1`) or down/not up (`0`).

This notebook uses the common group-preprocessed datasets. Model selection is performed with the training and validation data; the test set is used only once for final evaluation.

## 1. Import libraries and configure the experiment

In [ ]:
from pathlib import Path
import json
import shutil
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
OUTPUT_DIR = Path('IT25102978_Logistic_Regression_Results')
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)
print('Setup completed.')

## 2. Upload the three selected datasets

In [ ]:
required_files = ['train_selected.csv', 'validation_selected.csv', 'test_selected.csv']
missing_files = [name for name in required_files if not Path(name).exists()]

if missing_files:
    try:
        from google.colab import files
        print('Upload:', ', '.join(missing_files))
        files.upload()
    except ImportError:
        print('Place the three CSV files in the same folder as this notebook.')

still_missing = [name for name in required_files if not Path(name).exists()]
if still_missing:
    raise FileNotFoundError(f'Missing files: {still_missing}')
print('All required datasets are available.')

## 3. Load and validate the datasets

In [ ]:
train_df = pd.read_csv('train_selected.csv')
val_df = pd.read_csv('validation_selected.csv')
test_df = pd.read_csv('test_selected.csv')

datasets = {'Training': train_df, 'Validation': val_df, 'Testing': test_df}
expected_columns = list(train_df.columns)

for name, df in datasets.items():
    assert list(df.columns) == expected_columns, f'{name} columns do not match training columns.'
    assert 'Target' in df.columns, f'Target is missing from {name} data.'
    assert df.isna().sum().sum() == 0, f'{name} data contains missing values.'
    assert set(df['Target'].unique()).issubset({0, 1}), f'{name} Target must be binary.'
    print(f'{name:10s}: {df.shape[0]:,} rows, {df.shape[1]} columns, missing = {df.isna().sum().sum()}')

display(train_df.head())
print('Selected features:', [c for c in expected_columns if c != 'Target'])

## 4. Inspect the target distribution

In [ ]:
distribution_rows = []
for name, df in datasets.items():
    counts = df['Target'].value_counts().sort_index()
    distribution_rows.append({
        'Dataset': name,
        'Down_or_Not_Up_0': int(counts.get(0, 0)),
        'Up_1': int(counts.get(1, 0)),
        'Up_Percentage': round(100 * (df['Target'] == 1).mean(), 2)
    })
target_distribution = pd.DataFrame(distribution_rows)
display(target_distribution)
target_distribution.to_csv(OUTPUT_DIR / 'target_distribution.csv', index=False)

plot_df = target_distribution.set_index('Dataset')[['Down_or_Not_Up_0', 'Up_1']]
ax = plot_df.plot(kind='bar', figsize=(8, 5), color=['#d95f5f', '#4c9f70'])
ax.set_title('Target Distribution by Dataset')
ax.set_xlabel('Dataset')
ax.set_ylabel('Number of records')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / '01_target_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Separate features and target

In [ ]:
FEATURES = [c for c in train_df.columns if c != 'Target']
X_train, y_train = train_df[FEATURES], train_df['Target']
X_val, y_val = val_df[FEATURES], val_df['Target']
X_test, y_test = test_df[FEATURES], test_df['Target']

print('X_train:', X_train.shape, '| y_train:', y_train.shape)
print('X_val:  ', X_val.shape, '| y_val:  ', y_val.shape)
print('X_test: ', X_test.shape, '| y_test: ', y_test.shape)

## 6. Train a baseline Logistic Regression model

Logistic Regression estimates the probability of the positive class using the logistic (sigmoid) function. It is suitable as an interpretable baseline for binary classification.

In [ ]:
def calculate_metrics(y_true, y_pred, y_probability):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_true, y_probability)
    }

baseline_model = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
baseline_val_pred = baseline_model.predict(X_val)
baseline_val_prob = baseline_model.predict_proba(X_val)[:, 1]
baseline_metrics = calculate_metrics(y_val, baseline_val_pred, baseline_val_prob)

print('Baseline validation metrics')
display(pd.DataFrame([baseline_metrics]).round(4))
print(classification_report(y_val, baseline_val_pred, target_names=['Down/Not Up', 'Up']))

## 7. Perform 5-fold cross-validation on the training set

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_result = cross_validate(baseline_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_scores = pd.DataFrame({
    'Fold': np.arange(1, 6),
    'Accuracy': cv_result['test_accuracy'],
    'Precision': cv_result['test_precision'],
    'Recall': cv_result['test_recall'],
    'F1_Score': cv_result['test_f1'],
    'ROC_AUC': cv_result['test_roc_auc']
})
display(cv_scores.round(4))
display(cv_scores.drop(columns='Fold').agg(['mean', 'std']).round(4))
cv_scores.to_csv(OUTPUT_DIR / 'cross_validation_scores.csv', index=False)

## 8. Tune hyperparameters with GridSearchCV

Only the training set is used during tuning. The validation set is then used to compare the baseline and tuned models.

In [ ]:
parameter_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    param_grid=parameter_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)
grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best mean CV F1 score:', round(grid_search.best_score_, 4))

tuning_results = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
tuning_results.to_csv(OUTPUT_DIR / 'hyperparameter_tuning_results.csv', index=False)
display(tuning_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))

## 9. Evaluate the tuned model on the validation set

In [ ]:
tuned_model = grid_search.best_estimator_
tuned_val_pred = tuned_model.predict(X_val)
tuned_val_prob = tuned_model.predict_proba(X_val)[:, 1]
tuned_metrics = calculate_metrics(y_val, tuned_val_pred, tuned_val_prob)

validation_comparison = pd.DataFrame([
    {'Model': 'Baseline Logistic Regression', **baseline_metrics},
    {'Model': 'Tuned Logistic Regression', **tuned_metrics}
])
display(validation_comparison.round(4))
validation_comparison.to_csv(OUTPUT_DIR / 'validation_model_comparison.csv', index=False)

ax = validation_comparison.set_index('Model').plot(kind='bar', figsize=(10, 5), ylim=(0, 1), colormap='viridis')
ax.set_title('Baseline vs Tuned Model — Validation Set')
ax.set_ylabel('Score')
ax.tick_params(axis='x', rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_DIR / '02_validation_model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Train the final model and evaluate once on the test set

After selecting the tuned configuration, training and validation data are combined. The untouched test set provides the final unbiased holdout evaluation.

In [ ]:
X_train_final = pd.concat([X_train, X_val], ignore_index=True)
y_train_final = pd.concat([y_train, y_val], ignore_index=True)

final_model = LogisticRegression(
    **grid_search.best_params_,
    max_iter=3000,
    random_state=RANDOM_STATE
)
final_model.fit(X_train_final, y_train_final)
test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = calculate_metrics(y_test, test_pred, test_prob)

print('Final test metrics')
display(pd.DataFrame([test_metrics]).round(4))
print(classification_report(y_test, test_pred, target_names=['Down/Not Up', 'Up']))

pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / 'final_test_metrics.csv', index=False)
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as file:
    file.write(classification_report(y_test, test_pred, target_names=['Down/Not Up', 'Up']))

## 11. Visualize final model performance

In [ ]:
cm = confusion_matrix(y_test, test_pred)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Down/Not Up', 'Up'], yticklabels=['Down/Not Up', 'Up'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted class')
axes[0].set_ylabel('Actual class')

fpr, tpr, _ = roc_curve(y_test, test_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['ROC_AUC']:.3f}", color='#2a6fbb')
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

pr_precision, pr_recall, _ = precision_recall_curve(y_test, test_prob)
axes[2].plot(pr_recall, pr_precision, color='#7b3294')
axes[2].set_title('Precision–Recall Curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')

plt.tight_layout()
plt.savefig(PLOT_DIR / '03_final_evaluation_curves.png', dpi=200, bbox_inches='tight')
plt.show()

## 12. Interpret feature coefficients

In [ ]:
coefficient_table = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': final_model.coef_[0]
})
coefficient_table['Absolute_Coefficient'] = coefficient_table['Coefficient'].abs()
coefficient_table = coefficient_table.sort_values('Absolute_Coefficient', ascending=False)
display(coefficient_table)
coefficient_table.to_csv(OUTPUT_DIR / 'feature_coefficients.csv', index=False)

plot_coefficients = coefficient_table.sort_values('Coefficient')
colors = ['#d95f5f' if value < 0 else '#4c9f70' for value in plot_coefficients['Coefficient']]
plt.figure(figsize=(9, 5))
plt.barh(plot_coefficients['Feature'], plot_coefficients['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Feature Coefficients')
plt.xlabel('Coefficient')
plt.tight_layout()
plt.savefig(PLOT_DIR / '04_feature_coefficients.png', dpi=200, bbox_inches='tight')
plt.show()

## 13. Display sample predictions

In [ ]:
prediction_results = X_test.copy()
prediction_results['Actual_Target'] = y_test.to_numpy()
prediction_results['Predicted_Target'] = test_pred
prediction_results['Probability_Up'] = test_prob
prediction_results['Prediction_Correct'] = prediction_results['Actual_Target'] == prediction_results['Predicted_Target']
prediction_results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
display(prediction_results.head(15))

## 14. Save the trained model and experiment summary

In [ ]:
joblib.dump(final_model, OUTPUT_DIR / 'IT25102978_logistic_regression_model.joblib')

experiment_summary = {
    'student_id': 'IT25102978',
    'algorithm': 'Logistic Regression',
    'problem_type': 'Binary classification',
    'target_definition': '1 if the next closing price is higher than the current closing price; otherwise 0',
    'features': FEATURES,
    'training_rows': int(len(train_df)),
    'validation_rows': int(len(val_df)),
    'testing_rows': int(len(test_df)),
    'best_parameters': grid_search.best_params_,
    'best_cross_validation_f1': float(grid_search.best_score_),
    'baseline_validation_metrics': {k: float(v) for k, v in baseline_metrics.items()},
    'tuned_validation_metrics': {k: float(v) for k, v in tuned_metrics.items()},
    'final_test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'limitations': [
        'Historical market patterns do not guarantee future performance.',
        'The selected features contain limited short-term market context.',
        'Logistic Regression assumes a linear relationship in the log-odds.',
        'Transaction costs, news, and macroeconomic events are not included.'
    ]
}
with open(OUTPUT_DIR / 'experiment_summary.json', 'w') as file:
    json.dump(experiment_summary, file, indent=2)

print(json.dumps(experiment_summary, indent=2))

## 15. Conclusion

Use the produced metrics to complete this conclusion after running the notebook:

- Compare baseline and tuned validation F1 scores.
- Report final test Accuracy, Precision, Recall, F1 and ROC-AUC.
- Explain whether performance is meaningfully better than the majority-class baseline.
- Discuss the coefficient directions and the stated limitations.

## 16. Create and download the results ZIP

In [ ]:
zip_path = shutil.make_archive('IT25102978_Logistic_Regression_Results', 'zip', OUTPUT_DIR)
print('Created:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Download the ZIP from the notebook folder:', zip_path)